# Etapa 2 · Exploración de las fuentes de datos
**Reto Auditoría FEMSA - Analítico de descarga de agua** · TI2006B Gpo 303 · Equipo 1

Este cuaderno importa las 2 fuentes que compartió FEMSA, las carga a una base de datos SQL (SQLite) y las explora archivo por archivo y columna por columna con pandas y SQL.

| Fuente | Archivo | Contenido |
|---|---|---|
| Bitácora de descargas | `Bitacora descargas 2026.xlsx` | Formato CONAGUA 10-A, una hoja por planta, lecturas diarias del medidor en m³ |
| Reporting Factory | `ReportingFactory Ene26 - Oct26.xlsx` | Exportación del Power BI corporativo, una hoja por planta + TOTAL, descarga mensual en litros |

## 1. Configuración

In [ ]:
# Librerías
import sqlite3, datetime, calendar
# sqlite3: base de datos SQL ligera que viene con Python
# datetime: manejo de fechas
# calendar: saber cuántos días tiene cada mes
import pandas as pd # pandas: manejo de tablas (DataFrames)
import openpyxl # openpyxl: leer el Excel celda por celda, incluyendo encabezados, fórmulas y formato

# Nombres de los archivos de entrada
ARCHIVO_BITACORA = "Bitacora descargas 2026.xlsx" # lecturas diarias por planta (m³)
ARCHIVO_RF = "ReportingFactory Ene26 - Oct26.xlsx" # descarga mensual oficial (litros)

# Opciones de visualización
pd.set_option("display.width", 200) # ancho máximo de la tabla antes de cortarse
pd.set_option("display.max_columns", 30) # mostrar hasta 30 columnas
pd.set_option("display.max_rows", 120) # mostrar hasta 120 filas

# Base de datos SQL
# se crea una base de datos en memoria (no genera archivo)
# ahí cargaremos las tablas para explorarlas con consultas SQL
con = sqlite3.connect(":memory:")
def sql(q):
    """Ejecuta una consulta SQL y regresa el resultado como tabla."""
    # recibe el texto de la consulta (q), la corre en la base y devuelve el resultado como DataFrame para verlo como tabla
    return pd.read_sql_query(q, con)

## 2. Inspección de la estructura original de los archivos
Antes de importar revisamos cómo vienen los archivos: hojas, dimensiones y encabezados.

In [ ]:
# Abrir los archivos de Excel
# la bitácora se abre 2 veces porque sus columnas de "DIFERENCIA" son fórmulas:
wb_bit = openpyxl.load_workbook(ARCHIVO_BITACORA, data_only=True) # data_only=True: lee el RESULTADO de las fórmulas (los números)
wb_bitf = openpyxl.load_workbook(ARCHIVO_BITACORA) # sin data_only: lee la FÓRMULA escrita en la celda, para revisar contra qué celda resta cada día
wb_rf = openpyxl.load_workbook(ARCHIVO_RF, data_only=True) # Reporting Factory: solo nos interesan los valores

# ¿Cuántas hojas trae cada archivo?
print("Bitácora - hojas:", len(wb_bit.sheetnames));  print(wb_bit.sheetnames, "\n") # en la bitácora hay una hoja por planta (23)
print("Reporting Factory – hojas:", len(wb_rf.sheetnames)); print(wb_rf.sheetnames, "\n") # en Reporting Factory hay una hoja por planta más una hoja TOTAL (24)

# Revisar la estructura de una hoja de ejemplo
ws = wb_bit["Altamira"] # tomamos Altamira para ver cómo está armado el formato de CONAGUA antes de decidir cómo importarlo
print("Ejemplo hoja de bitácora (Altamira):", ws.max_row, "filas x", ws.max_column, "columnas")
# imprimimos las celdas clave del encabezado:
#    K1, K3   → título del formato de CONAGUA
#    C8       → etiqueta de la lectura inicial
#    G8, H8   → "EFLUENTE" y la lectura inicial del medidor de descarga (m³)
#    J8, K8   → valor de referencia diario (m³/día); lo usa el formato condicional para marcar en rojo; suponemos que es el límite autorizado
#    G9, H9   → "RIEGO" y la lectura inicial del medidor de riego
#    B11      → fecha del primer bloque mensual
#    B12, C12 → nombres de las columnas de cada bloque: FLUJO y DIFERENCIA
for celda in ["K1", "K3", "C8", "G8", "H8", "J8", "K8", "G9", "H9", "B11", "B12", "C12"]: print(f"  {celda}: {ws[celda].value}")

Bitácora - hojas: 23
['Altamira', 'Apizaco', 'Cayaco', 'Celaya', 'Cd. Victoria', 'Coatepec', 'Cuautitlán', 'Cuernavaca', 'Ixtacomitán', 'León', 'Morelia', 'Ojuelos', 'Pacífico', 'PozaRica', 'Querétaro', 'Reyes', 'Sabino', 'SanCristobal', 'SanJuanDelRio', 'Tapachula', 'Toluca', 'Tuxtla', 'Veracruz'] 

Reporting Factory – hojas: 24
['TOTAL', 'Altamira', 'Apizaco', 'Cayaco', 'Celaya', 'Cd. Victoria', 'Coatepec', 'Cuautitlán', 'Cuernavaca', 'Ixtacomitán', 'León', 'Morelia', 'Ojuelos', 'Pacífico', 'PozaRica', 'Querétaro', 'Reyes', 'Sabino', 'SanCristobal', 'SanJuanDelRio', 'Tapachula', 'Toluca', 'Tuxtla', 'Veracruz'] 

Ejemplo hoja de bitácora (Altamira): 59 filas x 60 columnas
  K1: COMISION NACIONAL DEL AGUA
  K3: "REPORTE DE LECTURAS DE MEDIDOR"       10-A
  C8: FLUJO DE DESCARGA AL 31 DE OCT-2026
  G8: EFLUENTE
  H8: 1041875
  J8: 1316
  K8: m3/día
  G9: RIEGO
  H9: 0
  B11: 2026-01-01 00:00:00
  B12: FLUJO
  C12: DIFERENCIA


In [ ]:
# Reporting Factory: encabezados de cada hoja
# el objetivo es confirmar que todas las hojas tienen las mismas columnas antes de juntarlas
#    for h in wb_rf.sheetnames → recorre cada hoja
#    wb_rf[h][1]               → toma la fila 1 de esa hoja (los encabezados)
#    [c.value for c in ...]    → saca el texto de cada celda de esa fila
#    (h, *[...])               → arma una fila con el nombre de la hoja + sus 4 encabezados
pd.DataFrame([(h, *[c.value for c in wb_rf[h][1]]) for h in wb_rf.sheetnames],
             columns=["hoja", "col_A", "col_B", "col_C", "col_D"])

,hoja,col_A,col_B,col_C,col_D
0,TOTAL,KOF DIM_TIME[NombreMes],KOF DIM_TIME[Month],KOF DIM_TIME[Año],[Agua_Descarga_PTAR_LITROS]
1,Altamira,KOF DIM_TIME[NombreMes],KOF DIM_TIME[Month],KOF DIM_TIME[Año],[Agua_Descarga_PTAR]
2,Apizaco,KOF DIM_TIME[NombreMes],KOF DIM_TIME[Month],KOF DIM_TIME[Año],[Agua_Descarga_PTAR]
3,Cayaco,KOF DIM_TIME[NombreMes],KOF DIM_TIME[Month],KOF DIM_TIME[Año],[Agua_Descarga_PTAR]
4,Celaya,KOF DIM_TIME[NombreMes],KOF DIM_TIME[Month],KOF DIM_TIME[Año],[Agua_Descarga_PTAR]
5,Cd. Victoria,KOF DIM_TIME[NombreMes],KOF DIM_TIME[Month],KOF DIM_TIME[Año],[Agua_Descarga_PTAR]
6,Coatepec,KOF DIM_TIME[NombreMes],KOF DIM_TIME[Month],KOF DIM_TIME[Año],[Agua_Descarga_PTAR]
7,Cuautitlán,KOF DIM_TIME[NombreMes],KOF DIM_TIME[Month],KOF DIM_TIME[Año],[Agua_Descarga_PTAR]
8,Cuernavaca,KOF DIM_TIME[NombreMes],KOF DIM_TIME[Month],KOF DIM_TIME[Año],[Agua_Descarga_PTAR]
9,Ixtacomitán,KOF DIM_TIME[NombreMes],KOF DIM_TIME[Month],KOF DIM_TIME[Año],[Agua_Descarga_PTAR]


## 3. Importación de Reporting Factory a SQL
Se toma cada hoja de planta (sin TOTAL) y se apilan en una tabla larga `rf_mensual`. La hoja TOTAL se guarda aparte para validarla.

In [ ]:
# Extracción de Reporting Factory
# junta todas las hojas en una sola tabla "larga": una fila por planta y mes
def extraer_reporting_factory(wb):
    """Apila todas las hojas de Reporting Factory en una tabla larga."""
    filas = [] # aquí se van guardando los registros
    for hoja in wb.sheetnames: # recorre TOTAL y las 23 plantas
        # iter_rows(values_only=True) lee cada fila como una lista de valores
        # [1:] se salta la fila 1, que son los encabezados
        for r in list(wb[hoja].iter_rows(values_only=True))[1:]:
            # cada fila trae nombre del mes, número de mes, año y litros
            filas.append({"hoja": hoja, "nombre_mes": r[0], "mes": r[1], "anio": r[2], "descarga_litros": r[3]})
    rf_raw = pd.DataFrame(filas) # convierte la lista en tabla
    # separamos en 2 tablas
    # rf_mensual → solo las plantas; la columna "hoja" pasa a llamarse "planta"
    rf_mensual = rf_raw[rf_raw.hoja != "TOTAL"].rename(columns={"hoja": "planta"})
    # rf_total → solo la hoja TOTAL, para después validar que sea la suma de las plantas
    rf_total   = rf_raw[rf_raw.hoja == "TOTAL"].drop(columns="hoja")
    return rf_mensual, rf_total

# ejecutamos la función con el archivo que ya abrimos
rf_mensual, rf_total = extraer_reporting_factory(wb_rf)

# Carga a la base de datos SQL
# to_sql guarda cada tabla en la base SQLite para poder consultarla con SQL
# index=False evita guardar el número de fila de pandas como columna
# if_exists="replace" permite volver a correr la celda sin error si la tabla ya existe
rf_mensual.to_sql("rf_mensual", con, index=False, if_exists="replace")
rf_total.to_sql("rf_total", con, index=False, if_exists="replace")
# primera consulta: mostramos los primeros 12 registros para confirmar que la tabla se cargó bien (planta, mes, año y litros)
sql("SELECT * FROM rf_mensual LIMIT 12")

,planta,nombre_mes,mes,anio,descarga_litros
0,Altamira,Enero,1,2026,24985000
1,Altamira,Febrero,2,2026,26295000
2,Altamira,Marzo,3,2026,27428000
3,Altamira,Abril,4,2026,27947000
4,Altamira,Mayo,5,2026,30827000
5,Altamira,Junio,6,2026,29899000
6,Altamira,Julio,7,2026,27881000
7,Altamira,Agosto,8,2026,26801000
8,Altamira,Septiembre,9,2026,31623000
9,Altamira,Octubre,10,2026,30607000


## 4. Importación de la bitácora a SQL
La bitácora no viene como tabla: cada mes es un bloque de 4 columnas (flujo y diferencia de efluente, flujo y diferencia de riego) y la planta solo aparece como nombre de la hoja. La transformamos a 2 tablas:

* `bit_encabezado`: una fila por planta con los datos de la cabecera (lectura inicial, límite diario, etiqueta de fecha).
* `bit_diaria`: una fila por planta y día, con la lectura acumulada del medidor y la diferencia que viene capturada en el archivo.

In [ ]:
# Extracción de la bitácora
def extraer_bitacora(wb_bit, wb_bitf):
    """Convierte el formato CONAGUA (bloques por mes) en 2 tablas: encabezado y lecturas diarias."""
    enc, dias = [], [] # listas donde se guardan los registros
    for hoja in wb_bit.sheetnames: # recorre las 23 plantas
        ws, wsf = wb_bit[hoja], wb_bitf[hoja] # ws: valores, wsf: fórmulas de la misma hoja
        # encabezado de la planta (celdas fijas del formato CONAGUA)
        enc.append({"planta": hoja,
                    "etiqueta_lectura_inicial": ws["C8"].value, # texto
                    "lectura_inicial_efluente_m3": ws["H8"].value, # lectura del medidor antes del 1 de enero
                    "lectura_inicial_riego_m3": ws["H9"].value, # lectura inicial del medidor de riego
                    "limite_diario_m3": ws["J8"].value, # valor de referencia diario (suponemos que es el límite)
                    "unidad_limite": ws["K8"].value}) # unidad: m3/día
        # buscamos dónde empieza cada mes: en la fila 11 hay una fecha arriba de cada bloque
        for col in range(1, ws.max_column + 1):
            fecha = ws.cell(11, col).value
            if not isinstance(fecha, datetime.datetime):
                continue # solo columnas que abren un bloque mensual
            # las filas 16 a 46 son los días 1 a 31 del mes
            for fila in range(16, 47):
                dia = fila - 15
                # calendar.monthrange da cuántos días tiene el mes
                if dia > calendar.monthrange(fecha.year, fecha.month)[1]:
                    continue
                # cada bloque mensual tiene 4 columnas a partir de la fecha:
                #    col     → flujo de efluente (lectura acumulada del medidor)
                #    col + 1 → diferencia de efluente (descarga del día)
                #    col + 2 → flujo de riego
                #    col + 3 → diferencia de riego
                # y en col - 1 está el número de día que escribió la planta
                dias.append({"planta": hoja,
                             "fecha": datetime.date(fecha.year, fecha.month, dia).isoformat(), # fecha completa AAAA-MM-DD
                             "anio": fecha.year, "mes": fecha.month, "dia": dia,
                             "dia_capturado": ws.cell(fila, col - 1).value,
                             "lectura_efluente_m3": ws.cell(fila, col).value,
                             "diferencia_efluente_m3": ws.cell(fila, col + 1).value,
                             "lectura_riego_m3": ws.cell(fila, col + 2).value,
                             "diferencia_riego_m3": ws.cell(fila, col + 3).value,
                             "formula_diferencia": wsf.cell(fila, col + 1).value}) # fórmula original, para revisar contra qué celda resta
    bit_encabezado = pd.DataFrame(enc) # convierte las listas en tablas
    bit_diaria = pd.DataFrame(dias)
    # las celdas vacías vienen como texto "" (por la fórmula =IF(...,"",...));
    # las pasamos a número y lo que no se pueda convertir queda como nulo (NaN)
    for c in ["diferencia_efluente_m3", "diferencia_riego_m3", "lectura_efluente_m3", "lectura_riego_m3"]:
        bit_diaria[c] = pd.to_numeric(bit_diaria[c].replace("", None), errors="coerce")
    return bit_encabezado, bit_diaria
# ejecutamos la función
bit_encabezado, bit_diaria = extraer_bitacora(wb_bit, wb_bitf)
# Carga a la base de datos SQL
# if_exists="replace" permite volver a correr la celda sin error si la tabla ya existe
bit_encabezado.to_sql("bit_encabezado", con, index=False, if_exists="replace")
bit_diaria.to_sql("bit_diaria", con, index=False, if_exists="replace")
# tamaño de cada tabla (filas, columnas)
print("bit_encabezado:", bit_encabezado.shape, " bit_diaria:", bit_diaria.shape)
# primeros 8 registros para confirmar que la tabla se cargó bien
sql("SELECT planta, fecha, lectura_efluente_m3, diferencia_efluente_m3, lectura_riego_m3, diferencia_riego_m3 FROM bit_diaria LIMIT 8")

bit_encabezado: (23, 6)  bit_diaria: (8395, 11)


,planta,fecha,lectura_efluente_m3,diferencia_efluente_m3,lectura_riego_m3,diferencia_riego_m3
0,Altamira,2026-01-01,1042678.0,803.0,0.0,0.0
1,Altamira,2026-01-02,1043566.0,888.0,0.0,0.0
2,Altamira,2026-01-03,1044433.0,867.0,0.0,0.0
3,Altamira,2026-01-04,1045277.0,844.0,0.0,0.0
4,Altamira,2026-01-05,1045904.0,627.0,0.0,0.0
5,Altamira,2026-01-06,1046711.0,807.0,0.0,0.0
6,Altamira,2026-01-07,1047324.0,613.0,0.0,0.0
7,Altamira,2026-01-08,1048335.0,1011.0,0.0,0.0


## 5. Exploración con pandas
Estadísticas descriptivas de cada columna numérica y promedios por planta y por mes, con las funciones vistas en clase.

In [ ]:
# describe(): conteo, promedio, desviación, mínimo, cuartiles y máximo de cada columna numérica
rf_mensual.describe()

,mes,anio,descarga_litros
count,230.000000,230.0,2.300000e+02
mean,5.500000,2026.0,1.077140e+07
std,2.878546,0.0,1.147026e+07
min,1.000000,2026.0,2.640000e+02
25%,3.000000,2026.0,9.705000e+05
50%,5.500000,2026.0,6.607500e+06
75%,8.000000,2026.0,1.882550e+07
max,10.000000,2026.0,5.259500e+07


In [ ]:
# filtramos solo enero a octubre, porque noviembre y diciembre vienen vacíos en la bitácora
bit_10 = bit_diaria[bit_diaria.mes <= 10]
# describe() de las 4 columnas numéricas de la bitácora
bit_10[["lectura_efluente_m3", "diferencia_efluente_m3", "lectura_riego_m3", "diferencia_riego_m3"]].describe()

,lectura_efluente_m3,diferencia_efluente_m3,lectura_riego_m3,diferencia_riego_m3
count,6.923000e+03,6923.000000,6923.000000,6923.000000
mean,2.732921e+06,366.092301,4876.602484,2.204391
std,1.898407e+06,398.097935,9190.291136,12.734536
min,5.611900e+04,0.000000,0.000000,0.000000
25%,9.390970e+05,27.000000,0.000000,0.000000
50%,2.701883e+06,223.000000,0.000000,0.000000
75%,4.763634e+06,608.000000,5308.000000,0.000000
max,5.970883e+06,3534.000000,34012.000000,110.000000


In [ ]:
# mean(): promedio de una columna
# round(..., 1) redondea a 1 decimal para que se lea mejor

# descarga diaria promedio de la bitácora, en m³ (todas las plantas y todos los días juntos)
print("Descarga diaria promedio (todas las plantas):", round(bit_10["diferencia_efluente_m3"].mean(), 1), "m³")

# descarga mensual promedio de Reporting Factory, en litros (todas las plantas y meses juntos)
print("Descarga mensual promedio en Reporting Factory:", round(rf_mensual["descarga_litros"].mean()), "L")

Descarga diaria promedio (todas las plantas): 366.1 m³
Descarga mensual promedio en Reporting Factory: 10771397 L


In [ ]:
# groupby(): agrupa los registros por planta y calcula estadísticas de la descarga diaria de cada una
#    groupby("planta")                  → un grupo por cada una de las 23 plantas
#    ["diferencia_efluente_m3"]         → columna que vamos a resumir (descarga del día en m³)
#    .agg(["mean","min","max","count"]) → promedio, mínimo, máximo y número de días con lectura
#    .round(1)                          → redondea a 1 decimal
bit_10.groupby("planta")["diferencia_efluente_m3"].agg(["mean", "min", "max", "count"]).round(1)

,mean,min,max,count
planta,,,,
Altamira,935.2,0.0,1466.0,304
Apizaco,774.1,0.0,1156.0,304
Cayaco,266.9,0.0,433.0,304
Cd. Victoria,12.4,0.0,82.0,294
Celaya,24.8,0.0,46.0,304
Coatepec,327.8,0.0,2001.0,292
Cuautitlán,510.3,0.0,1345.0,298
Cuernavaca,174.3,0.0,796.0,300
Ixtacomitán,1433.7,0.0,2338.0,304


In [ ]:
# groupby(): descarga mensual de Reporting Factory por planta
#    groupby("planta")          → un grupo por planta
#    ["descarga_litros"]        → descarga mensual oficial en litros
#    .agg(["mean","min","max"]) → promedio, mínimo y máximo de los 10 meses
#    .round(0)                  → sin decimales
rf_mensual.groupby("planta")["descarga_litros"].agg(["mean", "min", "max"]).round(0)

,mean,min,max
planta,,,
Altamira,28429300.0,24985000,31623000
Apizaco,23835000.0,20922000,26118000
Cayaco,8112900.0,6159000,8945000
Cd. Victoria,370200.0,318000,442000
Celaya,970100.0,292000,1223000
Coatepec,8551722.0,10222,12100000
Cuautitlán,15168100.0,13204000,17701000
Cuernavaca,4997492.0,4918,6725000
Ixtacomitán,44057600.0,40676000,52595000


In [ ]:
# groupby() con 2 columnas: total de descarga de la bitácora por planta y mes, en m³
#    groupby(["planta", "mes"]) → un grupo por cada combinación planta-mes (23 x 10 = 230)
#    ["diferencia_efluente_m3"] → descarga del día
#    .sum()                     → suma los días para tener la descarga del mes
#    .unstack("mes")            → pasa los meses a columnas: queda una fila por planta y una columna por mes
bit_10.groupby(["planta", "mes"])["diferencia_efluente_m3"].sum().unstack("mes")

mes,1,2,3,4,5,6,7,8,9,10
planta,,,,,,,,,,
Altamira,24985.0,26294.0,27428.0,27947.0,30827.0,29899.0,27881.0,26797.0,31623.0,30607.0
Apizaco,23002.0,20922.0,24052.0,23756.0,23790.0,24353.0,22531.0,22594.0,24470.0,25844.0
Cayaco,7888.0,6159.0,8945.0,8569.0,8230.0,7657.0,7606.0,8782.0,8473.0,8825.0
Cd. Victoria,334.0,318.0,352.0,508.0,337.0,375.0,393.0,332.0,383.0,300.0
Celaya,937.0,292.0,495.0,649.0,1006.0,795.0,761.0,745.0,986.0,885.0
Coatepec,8436.0,7540.0,12098.0,7366.0,10738.0,10968.0,8420.0,9836.0,10999.0,9326.0
Cuautitlán,14299.0,13204.0,16007.0,16365.0,17701.0,13918.0,16325.0,14320.0,16372.0,13562.0
Cuernavaca,5286.0,4295.0,6630.0,6131.0,5077.0,4770.0,3427.0,5658.0,5241.0,5789.0
Ixtacomitán,42721.0,40865.0,42975.0,43815.0,40676.0,46838.0,44668.0,44210.0,47874.0,41213.0


## 6. Exploración de Reporting Factory con SQL

In [ ]:
# revisa cuántos registros hay, si están todas las plantas y meses, si hay nulos y el rango de valores
sql("""
SELECT COUNT(*) AS registros,                         -- total de filas (esperado: 23 plantas x 10 meses = 230)
       COUNT(DISTINCT planta) AS plantas,             -- plantas distintas (esperado: 23)
       COUNT(DISTINCT mes) AS meses,                  -- meses distintos (esperado: 10)
       MIN(mes) AS mes_min, MAX(mes) AS mes_max,      -- rango de meses (enero = 1 a octubre = 10)
       COUNT(DISTINCT anio) AS anios,                 -- años distintos (esperado: solo 2026)
       SUM(descarga_litros IS NULL) AS litros_nulos,  -- cuántos meses no tienen valor
       MIN(descarga_litros) AS litros_min,            -- descarga mínima registrada
       MAX(descarga_litros) AS litros_max,            -- descarga máxima registrada
       ROUND(AVG(descarga_litros)) AS litros_promedio -- promedio de descarga mensual
FROM rf_mensual
""")

,registros,plantas,meses,mes_min,mes_max,anios,litros_nulos,litros_min,litros_max,litros_promedio
0,230,23,10,1,10,1,0,264,52595000,10771397.0


In [ ]:
# Registros por planta y posibles duplicados planta-mes
sql("""
SELECT planta,
       COUNT(*) AS meses,                      -- total de registros de la planta (esperado: 10)
       COUNT(DISTINCT mes) AS meses_distintos, -- meses diferentes; si es igual a "meses", no hay duplicados
       MIN(descarga_litros) AS min_litros,     -- mes con menor descarga
       MAX(descarga_litros) AS max_litros      -- mes con mayor descarga
FROM rf_mensual
GROUP BY planta                                -- un renglón por planta
ORDER BY planta
""")

,planta,meses,meses_distintos,min_litros,max_litros
0,Altamira,10,10,24985000,31623000
1,Apizaco,10,10,20922000,26118000
2,Cayaco,10,10,6159000,8945000
3,Cd. Victoria,10,10,318000,442000
4,Celaya,10,10,292000,1223000
5,Coatepec,10,10,10222,12100000
6,Cuautitlán,10,10,13204000,17701000
7,Cuernavaca,10,10,4918,6725000
8,Ixtacomitán,10,10,40676000,52595000
9,León,10,10,22691,22224000


In [ ]:
# Valores que no terminan en 000
# las plantas miden en m³ y Reporting Factory lo pasa a litros multiplicando por 1,000, así que todo valor correcto debería terminar en 000;
# si un valor no termina en 000, probablemente no se convirtió y quedó en m³
sql("""
SELECT planta, nombre_mes, descarga_litros
FROM rf_mensual
WHERE descarga_litros % 1000 <> 0 -- % es el residuo de la división; si no es 0, no es múltiplo de 1,000
""")

,planta,nombre_mes,descarga_litros
0,Coatepec,Junio,10222
1,Cuernavaca,Junio,4918
2,León,Marzo,22691
3,Morelia,Junio,16628
4,Toluca,Marzo,23610
5,Tuxtla,Septiembre,264


In [ ]:
# ¿El TOTAL de Reporting Factory es la suma de las plantas?
# compara, mes por mes, el valor de la hoja TOTAL contra la suma de las 23 hojas de planta
sql("""
SELECT t.mes, t.nombre_mes,
       t.descarga_litros AS total_hoja,                         -- valor que trae la hoja TOTAL
       SUM(p.descarga_litros) AS suma_plantas,                  -- suma de las 23 plantas en ese mes
       t.descarga_litros - SUM(p.descarga_litros) AS diferencia -- 0 = cuadra
FROM rf_total t                                                 -- t = hoja TOTAL
JOIN rf_mensual p ON p.mes = t.mes                              -- p = plantas; se unen por mes
GROUP BY t.mes
ORDER BY t.mes
""")

,mes,nombre_mes,total_hoja,suma_plantas,diferencia
0,1,Enero,238957000,238957000,0
1,2,Febrero,229070000,229070000,0
2,3,Marzo,218742301,218742301,0
3,4,Abril,257535000,257535000,0
4,5,Mayo,264841000,264841000,0
5,6,Junio,227444768,227444768,0
6,7,Julio,243850000,243850000,0
7,8,Agosto,253978000,253978000,0
8,9,Septiembre,274767264,274767264,0
9,10,Octubre,268236000,268236000,0


## 7. Exploración de la bitácora con SQL

In [ ]:
# Exploración de la bitácora con SQL: encabezado de cada hoja
# muestra los datos fijos de cada planta que vienen arriba del formato CONAGUA:
#    etiqueta_lectura_inicial    → texto que indica a qué fecha corresponde la lectura inicial
#    lectura_inicial_efluente_m3 → lectura del medidor de descarga antes del 1 de enero
#    lectura_inicial_riego_m3    → lectura inicial del medidor de riego
#    limite_diario_m3            → valor de referencia diario (celda J8); suponemos que es el límite autorizado
sql("SELECT * FROM bit_encabezado")

,planta,etiqueta_lectura_inicial,lectura_inicial_efluente_m3,lectura_inicial_riego_m3,limite_diario_m3,unidad_limite
0,Altamira,FLUJO DE DESCARGA AL 31 DE OCT-2026,1041875,0,1316.00,m3/día
1,Apizaco,FLUJO DE DESCARGA AL 31 DE DIC-2025,4556771,27299,1100.38,m3/día
2,Cayaco,FLUJO DE DESCARGA AL 31 DE DIC-2025,2806041,0,388.46,m3/día
3,Celaya,FLUJO DE DESCARGA AL 31 DE DIC-2025,3229654,4120,48.53,m3/día
4,Cd. Victoria,FLUJO DE DESCARGA AL 31 DE DIC-2025,3955254,0,17.23,m3/día
5,Coatepec,FLUJO DE DESCARGA AL 31 DE DIC-2025,4735448,0,465.90,m3/día
6,Cuautitlán,FLUJO DE DESCARGA AL 31 DE DIC-2025,826948,0,721.82,m3/día
7,Cuernavaca,FLUJO DE DESCARGA AL 31 DE DIC-2025,5250535,6640,252.02,m3/día
8,Ixtacomitán,FLUJO DE DESCARGA AL 31 DE DIC-2025,2592030,0,2010.63,m3/día
9,León,FLUJO DE DESCARGA AL 31 DE DIC-2025,5214131,18450,952.89,m3/día


In [ ]:
# Perfil general de la bitácora (enero a octubre)
# revisa si están todos los días, si faltan lecturas y el rango de la descarga diaria
sql("""
SELECT COUNT(*) AS dias_esperados,                                -- 23 plantas x 304 días = 6,992
       COUNT(DISTINCT planta) AS plantas,                         -- esperado: 23
       MIN(fecha) AS fecha_min, MAX(fecha) AS fecha_max,          -- rango: 1 de enero a 31 de octubre
       SUM(lectura_efluente_m3 IS NULL) AS lecturas_vacias,       -- días sin lectura del medidor
       SUM(diferencia_efluente_m3 IS NULL) AS diferencias_vacias, -- días sin descarga calculada
       SUM(dia_capturado <> dia) AS dias_mal_numerados,           -- revisa que el día escrito por la planta coincida con la fila
       MIN(diferencia_efluente_m3) AS dif_min,                    -- descarga diaria mínima
       MAX(diferencia_efluente_m3) AS dif_max,                    -- descarga diaria máxima
       ROUND(AVG(diferencia_efluente_m3),1) AS dif_promedio       -- descarga diaria promedio
FROM bit_diaria
WHERE mes <= 10                                                   -- noviembre y diciembre vienen vacíos
""")

,dias_esperados,plantas,fecha_min,fecha_max,lecturas_vacias,diferencias_vacias,dias_mal_numerados,dif_min,dif_max,dif_promedio
0,6992,23,2026-01-01,2026-10-31,69,69,0,0.0,3534.0,366.1


In [ ]:
# Lecturas por mes: confirma qué meses tienen datos
# compara los días que tiene cada mes (de las 23 plantas juntas) contra los días que sí tienen lectura
sql("""
SELECT mes,
      COUNT(*) AS dias,                                   -- días posibles del mes x 23 plantas
      SUM(lectura_efluente_m3 IS NOT NULL) AS con_lectura -- días que sí tienen lectura del medidor
FROM bit_diaria
GROUP BY mes
ORDER BY mes
""")

,mes,dias,con_lectura
0,1,713,709
1,2,644,640
2,3,713,708
3,4,690,680
4,5,713,710
5,6,690,686
6,7,713,709
7,8,713,704
8,9,690,683
9,10,713,694


In [ ]:
# Días sin lectura del medidor (enero–octubre)
# muestra qué plantas y en qué meses tienen días vacíos, y en qué días empieza y termina el hueco
sql("""
SELECT planta, mes,
       COUNT(*) AS dias_sin_lectura, -- cuántos días del mes no tienen lectura
       MIN(dia) AS desde_dia,        -- primer día vacío
       MAX(dia) AS hasta_dia         -- último día vacío
FROM bit_diaria
WHERE mes <= 10                      -- solo enero a octubre
  AND lectura_efluente_m3 IS NULL    -- solo los días sin lectura
GROUP BY planta, mes
ORDER BY planta, mes
""")

,planta,mes,dias_sin_lectura,desde_dia,hasta_dia
0,Cd. Victoria,3,5,27,31
1,Cd. Victoria,10,5,27,31
2,Coatepec,5,3,29,31
3,Coatepec,8,5,27,31
4,Coatepec,9,4,27,30
5,Cuautitlán,6,2,29,30
6,Cuautitlán,10,4,28,31
7,Cuernavaca,7,4,28,31
8,León,8,4,28,31
9,Morelia,9,3,28,30


In [ ]:
# Columna de riego: ¿qué plantas sí la usan?
# si el medidor de riego no se mueve en todo el periodo (una sola lectura distinta), la planta no usa riego
sql("""
SELECT planta,
       COUNT(DISTINCT lectura_riego_m3) AS lecturas_distintas,  -- 1 = el medidor nunca cambió
       SUM(COALESCE(diferencia_riego_m3, 0)) AS riego_total_m3  -- agua total de riego ene-oct
                                                                -- COALESCE cambia los vacíos por 0 para poder sumar
FROM bit_diaria
WHERE mes <= 10
GROUP BY planta
ORDER BY riego_total_m3 DESC
""")

,planta,lecturas_distintas,riego_total_m3
0,Celaya,38,2563.0
1,SanJuanDelRio,32,2142.0
2,Cuernavaca,31,2100.0
3,Morelia,29,1885.0
4,Querétaro,31,1841.0
5,Apizaco,26,1744.0
6,León,25,1623.0
7,Ojuelos,20,1363.0
8,Veracruz,1,0.0
9,Tuxtla,1,0.0


8. Reglas de negocio y corrección con SQL

In [ ]:
# ======================================================================
# A.8 Reglas de negocio y correccion (SQLite)
# Pegar en una celda del cuaderno DESPUES de A.3 y A.4: usa las tablas
# bit_encabezado, bit_diaria y rf_mensual que ya estan en la conexion `con`.
# executescript permite correr varias sentencias SQL seguidas.
# ======================================================================
con.executescript("""
-- =====================================================================
-- Reto FEMSA | Etapa 2-3: Reglas de negocio y correccion (SQLite)
-- Se ejecuta en el mismo cuaderno de Colab, sobre las tablas que ya se
-- importaron en A.3 y A.4:
--     bit_encabezado, bit_diaria, rf_mensual, rf_total
-- Uso:  con.executescript(open("query_reglas_negocio_sqlite.sql").read())
-- Tablas que crea:
--     bit_dia_acond   (planta-dia)  atributos y banderas
--     conciliacion_mes (planta-mes) indicadores de las preguntas 2, 3 y 4
-- Los umbrales (k_pico, n_constante, rango de unidad) son supuestos
-- explicitos: se validan con los profesores y con Auditoria FEMSA.
-- =====================================================================

-- ---------------------------------------------------------------------
-- 0. PARAMETROS
-- ---------------------------------------------------------------------
DROP TABLE IF EXISTS parametros;
CREATE TABLE parametros AS
SELECT 10     AS mes_max,       -- R1: alcance ene-oct (nov-dic vienen vacios)
       1000.0 AS factor_m3_l,   -- R4: 1 m3 = 1,000 L
       1.5    AS k_pico,        -- R7: pico = Q3 + k*(Q3-Q1)
       5      AS n_constante,   -- R8: dias seguidos con la misma descarga
       0.5    AS tol_m3,        -- R3: tolerancia vs diferencia reportada
       900.0  AS ratio_min,     -- R10: rango de la razon bitacora(L)/RF(L)
       1100.0 AS ratio_max;     --     para marcar posible error de unidad


-- ---------------------------------------------------------------------
-- 1. LECTURAS OBSERVADAS + LECTURA INICIAL (R2, R3)
--    Solo dias con lectura. Se agrega la lectura inicial como ancla
--    del 31-dic-2025, para que el 1 de enero tenga lectura anterior.
--    La anterior es la ultima lectura que SI tiene valor (R3).
-- ---------------------------------------------------------------------
DROP TABLE IF EXISTS t_obs;
CREATE TABLE t_obs AS
WITH obs AS (
  SELECT planta, fecha, lectura_efluente_m3 AS lect, lectura_riego_m3 AS lect_rie
  FROM bit_diaria
  WHERE mes <= (SELECT mes_max FROM parametros) AND lectura_efluente_m3 IS NOT NULL
  UNION ALL
  SELECT planta, '2025-12-31', lectura_inicial_efluente_m3, lectura_inicial_riego_m3
  FROM bit_encabezado
)
SELECT planta, fecha, lect, lect_rie,
       LAG(lect)     OVER w AS lect_anterior,
       LAG(lect_rie) OVER w AS lect_rie_anterior,
       LAG(fecha)    OVER w AS fecha_anterior,
       CAST(julianday(fecha) - julianday(LAG(fecha) OVER w) AS INTEGER) AS dias_entre_lecturas
FROM obs
WINDOW w AS (PARTITION BY planta ORDER BY fecha);


-- ---------------------------------------------------------------------
-- 2. PLANTA-DIA BASE: diferencia calculada, descarga promedio por dia,
--    banderas simples (R1, R2, R3, R5, R6)
-- ---------------------------------------------------------------------
DROP TABLE IF EXISTS t_dia;
CREATE TABLE t_dia AS
SELECT
  d.planta, d.fecha, d.anio, d.mes, d.dia,
  d.lectura_efluente_m3                        AS lectura,
  o.lect_anterior                              AS lectura_anterior,
  o.dias_entre_lecturas,
  d.diferencia_efluente_m3                     AS diferencia_reportada,
  o.lect - o.lect_anterior                     AS diferencia_calculada,                 -- R3
  (o.lect - o.lect_anterior) * 1.0 / o.dias_entre_lecturas AS descarga_promedio_dia,    -- R3
  (o.lect_rie - o.lect_rie_anterior)           AS riego_calculado,
    d.lectura_riego_m3                           AS lectura_riego,
  d.diferencia_riego_m3                        AS diferencia_riego_reportada,
  CASE WHEN d.lectura_efluente_m3 IS NULL THEN 1 ELSE 0 END                AS flag_sin_lectura,   -- R2
  CASE WHEN o.lect < o.lect_anterior THEN 1 ELSE 0 END                     AS flag_retroceso,     -- R5
  CASE WHEN o.lect - o.lect_anterior = 0 THEN 1 ELSE 0 END                 AS flag_cero,          -- R6
  CASE WHEN COUNT(*) OVER (PARTITION BY d.planta, d.fecha) > 1 THEN 1 ELSE 0 END AS flag_duplicado, -- R6
  CASE WHEN ABS((o.lect - o.lect_anterior) - d.diferencia_efluente_m3)
            > (SELECT tol_m3 FROM parametros) THEN 1 ELSE 0 END            AS flag_difiere_reportada -- R3
FROM bit_diaria d
LEFT JOIN t_obs o ON o.planta = d.planta AND o.fecha = d.fecha
WHERE d.mes <= (SELECT mes_max FROM parametros);


-- ---------------------------------------------------------------------
-- 3. LIMITE DE PICO POR PLANTA (R7): Q3 + 1.5*(Q3-Q1) de la descarga
--    promedio por dia. Cuartiles con interpolacion lineal (igual que
--    pandas.quantile), porque SQLite no trae funcion de percentil.
-- ---------------------------------------------------------------------
DROP TABLE IF EXISTS t_cuartiles;
CREATE TABLE t_cuartiles AS
WITH ord AS (
  SELECT planta, descarga_promedio_dia AS x,
         ROW_NUMBER() OVER (PARTITION BY planta ORDER BY descarga_promedio_dia) AS rn,
         COUNT(*)     OVER (PARTITION BY planta)                                AS n
  FROM t_dia
  WHERE descarga_promedio_dia IS NOT NULL
),
pos AS (
  SELECT planta, n,
         1 + (n - 1) * 0.25 AS p1, 1 + (n - 1) * 0.75 AS p3
  FROM ord GROUP BY planta
),
q AS (
  SELECT p.planta,
    -- Q1
    MAX(CASE WHEN o.rn = CAST(p.p1 AS INT) THEN o.x END)
      + (p.p1 - CAST(p.p1 AS INT)) * (COALESCE(MAX(CASE WHEN o.rn = CAST(p.p1 AS INT) + 1 THEN o.x END),
                                                MAX(CASE WHEN o.rn = CAST(p.p1 AS INT) THEN o.x END))
                                      - MAX(CASE WHEN o.rn = CAST(p.p1 AS INT) THEN o.x END)) AS q1,
    -- Q3
    MAX(CASE WHEN o.rn = CAST(p.p3 AS INT) THEN o.x END)
      + (p.p3 - CAST(p.p3 AS INT)) * (COALESCE(MAX(CASE WHEN o.rn = CAST(p.p3 AS INT) + 1 THEN o.x END),
                                                MAX(CASE WHEN o.rn = CAST(p.p3 AS INT) THEN o.x END))
                                      - MAX(CASE WHEN o.rn = CAST(p.p3 AS INT) THEN o.x END)) AS q3
  FROM pos p JOIN ord o ON o.planta = p.planta
  GROUP BY p.planta
)
SELECT planta, q1, q3,
       q3 + (SELECT k_pico FROM parametros) * (q3 - q1) AS limite_pico
FROM q;


-- ---------------------------------------------------------------------
-- 4. VALOR CONSTANTE (R8): rachas de dias consecutivos con la misma
--    diferencia diaria (> 0). Se marcan todos los dias de la racha.
--    Un dia sin lectura interrumpe la racha.
-- ---------------------------------------------------------------------
DROP TABLE IF EXISTS t_racha;
CREATE TABLE t_racha AS
WITH a AS (
  SELECT planta, fecha, diferencia_calculada AS v,
         CASE WHEN diferencia_calculada IS NOT NULL
                   AND diferencia_calculada = LAG(diferencia_calculada) OVER (PARTITION BY planta ORDER BY fecha)
              THEN 0 ELSE 1 END AS nuevo
  FROM t_dia
),
b AS (
  SELECT planta, fecha, v,
         SUM(nuevo) OVER (PARTITION BY planta ORDER BY fecha ROWS UNBOUNDED PRECEDING) AS grp
  FROM a
)
SELECT planta, fecha,
       CASE WHEN v > 0 AND COUNT(v) OVER (PARTITION BY planta, grp) >= (SELECT n_constante FROM parametros)
            THEN 1 ELSE 0 END AS flag_constante
FROM b;


-- ---------------------------------------------------------------------
-- 5. TABLA PLANTA-DIA ACONDICIONADA (resultado final del nivel diario)
-- ---------------------------------------------------------------------
DROP TABLE IF EXISTS bit_dia_acond;
CREATE TABLE bit_dia_acond AS
SELECT
  t.*,
  c.limite_pico,
  CASE WHEN t.descarga_promedio_dia > c.limite_pico THEN 1 ELSE 0 END AS flag_pico,       -- R7
  r.flag_constante,                                                                       -- R8
  CASE WHEN t.flag_cero = 1 OR t.flag_duplicado = 1
         OR t.descarga_promedio_dia > c.limite_pico OR r.flag_constante = 1
       THEN 1 ELSE 0 END AS flag_anomalia                                                  -- R9
FROM t_dia t
JOIN t_cuartiles c ON c.planta = t.planta
JOIN t_racha     r ON r.planta = t.planta AND r.fecha = t.fecha;


-- ---------------------------------------------------------------------
-- 6. DESCARGA MENSUAL AJUSTADA (R12, propuesta): cada tramo entre dos
--    lecturas se reparte en partes iguales entre los dias que cubre, y
--    asi un hueco de fin de mes no se corre al mes siguiente.
-- ---------------------------------------------------------------------
DROP TABLE IF EXISTS t_reparto;
CREATE TABLE t_reparto AS
SELECT d.planta, d.mes,
       SUM((o.lect - o.lect_anterior) * 1.0 / o.dias_entre_lecturas)         AS descarga_ajustada_m3,
       SUM((o.lect_rie - o.lect_rie_anterior) * 1.0 / o.dias_entre_lecturas) AS riego_ajustado_m3
FROM t_obs o
JOIN bit_diaria d ON d.planta = o.planta AND d.fecha > o.fecha_anterior AND d.fecha <= o.fecha
WHERE o.fecha_anterior IS NOT NULL AND o.lect >= o.lect_anterior
  AND d.mes <= (SELECT mes_max FROM parametros)
GROUP BY d.planta, d.mes;


-- ---------------------------------------------------------------------
-- 7. RESUMEN BITACORA PLANTA-MES (indicadores de las preguntas 3 y 4)
-- ---------------------------------------------------------------------
DROP TABLE IF EXISTS t_bit_mes;
CREATE TABLE t_bit_mes AS
SELECT
  a.planta, a.anio, a.mes,
  COUNT(*)                                           AS dias_naturales,
  SUM(1 - a.flag_sin_lectura)                        AS dias_con_registro,
  COUNT(*) - SUM(1 - a.flag_sin_lectura)             AS dias_faltantes,
  SUM(a.diferencia_calculada)                        AS descarga_bit_m3,
  SUM(a.riego_calculado)                             AS riego_bit_m3,
  SUM(a.flag_cero)                                   AS dias_cero,
  SUM(a.flag_duplicado)                              AS dias_duplicado,
  SUM(a.flag_pico)                                   AS dias_pico,
  SUM(a.flag_constante)                              AS dias_constante,
  SUM(a.flag_anomalia)                               AS dias_anomalos,
  SUM(a.flag_retroceso)                              AS dias_retroceso,
  SUM(a.flag_difiere_reportada)                      AS dias_difiere_reportada
FROM bit_dia_acond a
GROUP BY a.planta, a.anio, a.mes;


-- ---------------------------------------------------------------------
-- 8. CONCILIACION PLANTA-MES (preguntas 2, 3 y 4)
-- ---------------------------------------------------------------------
DROP TABLE IF EXISTS conciliacion_mes;
CREATE TABLE conciliacion_mes AS
SELECT
  b.planta, b.anio, b.mes,
  b.planta || '-' || b.anio || '-' || printf('%02d', b.mes) AS periodo,
  b.dias_naturales, b.dias_con_registro, b.dias_faltantes,

  -- Pregunta 2 ------------------------------------------------------
  b.descarga_bit_m3 * p.factor_m3_l                         AS descarga_bit_L,         -- R4
  r.descarga_litros                                         AS rf_original_L,
  -- R10: posible error de unidad (RF no es multiplo de 1,000 y la razon cae en el rango)
  CASE WHEN r.descarga_litros % 1000 <> 0
        AND b.descarga_bit_m3 * p.factor_m3_l * 1.0 / r.descarga_litros BETWEEN p.ratio_min AND p.ratio_max
       THEN 1 ELSE 0 END                                    AS flag_posible_error_unidad,
  CASE WHEN r.descarga_litros % 1000 <> 0
        AND b.descarga_bit_m3 * p.factor_m3_l * 1.0 / r.descarga_litros BETWEEN p.ratio_min AND p.ratio_max
       THEN r.descarga_litros * 1000 ELSE r.descarga_litros END AS rf_L,                 -- R10
  -- Indicadores de tu documento (suma simple de las diferencias diarias)
  b.descarga_bit_m3 * p.factor_m3_l
    - CASE WHEN r.descarga_litros % 1000 <> 0
            AND b.descarga_bit_m3 * p.factor_m3_l * 1.0 / r.descarga_litros BETWEEN p.ratio_min AND p.ratio_max
           THEN r.descarga_litros * 1000 ELSE r.descarga_litros END AS diferencia_L,

  -- Variantes propuestas (R12, R13): reparto de huecos y riego
  rp.descarga_ajustada_m3 * p.factor_m3_l                   AS descarga_bit_ajustada_L,
  (rp.descarga_ajustada_m3 + rp.riego_ajustado_m3) * p.factor_m3_l AS descarga_mas_riego_ajustada_L,

  -- Pregunta 3 ------------------------------------------------------
  b.dias_cero, b.dias_duplicado, b.dias_pico, b.dias_constante, b.dias_anomalos,
  ROUND(b.dias_anomalos * 100.0 / NULLIF(b.dias_con_registro, 0), 2) AS pct_dias_anomalos,

  -- Pregunta 4 ------------------------------------------------------
  b.dias_retroceso,
  ROUND((b.dias_naturales - b.dias_retroceso - b.dias_faltantes) * 100.0 / b.dias_naturales, 2) AS pct_continuidad,
  b.dias_difiere_reportada
FROM t_bit_mes b
CROSS JOIN parametros p
JOIN rf_mensual r  ON r.planta = b.planta AND r.anio = b.anio AND r.mes = b.mes
LEFT JOIN t_reparto rp ON rp.planta = b.planta AND rp.mes = b.mes;

-- Porcentajes de discrepancia sobre RF (fuente de referencia), con la formula del documento:
-- (|diferencia mensual| / RF) x 100
ALTER TABLE conciliacion_mes ADD COLUMN pct_discrepancia REAL;
ALTER TABLE conciliacion_mes ADD COLUMN pct_discrepancia_ajustada REAL;
ALTER TABLE conciliacion_mes ADD COLUMN pct_discrepancia_ajustada_con_riego REAL;
UPDATE conciliacion_mes SET
  pct_discrepancia                     = ROUND(ABS(diferencia_L) * 100.0 / rf_L, 2),
  pct_discrepancia_ajustada            = ROUND(ABS(descarga_bit_ajustada_L - rf_L) * 100.0 / rf_L, 2),
  pct_discrepancia_ajustada_con_riego  = ROUND(ABS(descarga_mas_riego_ajustada_L - rf_L) * 100.0 / rf_L, 2)
WHERE rf_L > 0;

-- Version en m3 con 2 decimales (las columnas en litros se conservan tal cual)
ALTER TABLE conciliacion_mes ADD COLUMN descarga_bit_m3 REAL;
ALTER TABLE conciliacion_mes ADD COLUMN rf_m3 REAL;
ALTER TABLE conciliacion_mes ADD COLUMN diferencia_m3 REAL;
UPDATE conciliacion_mes SET
  descarga_bit_m3 = ROUND(descarga_bit_L / 1000.0, 2),
  rf_m3           = ROUND(rf_L / 1000.0, 2),
  diferencia_m3   = ROUND(diferencia_L / 1000.0, 2);

""")

# Verificaciones rapidas (usa la funcion sql() definida en A.1)

# 1) Tamano de las tablas nuevas: esperado 6,992 planta-dia y 230 planta-mes
sql("""SELECT (SELECT COUNT(*) FROM bit_dia_acond)   AS planta_dia,
              (SELECT COUNT(*) FROM conciliacion_mes) AS planta_mes""")

# 2) Banderas del nivel diario
sql("""SELECT SUM(flag_sin_lectura) AS dias_sin_lectura, SUM(flag_retroceso) AS retrocesos,
              SUM(flag_cero) AS ceros, SUM(flag_duplicado) AS duplicados,
              SUM(flag_pico) AS picos, SUM(flag_constante) AS constantes,
              SUM(flag_anomalia) AS dias_anomalos, SUM(flag_difiere_reportada) AS difiere_reportada
       FROM bit_dia_acond""")

# 3) Planta-meses con mayor discrepancia (indicador del documento)
sql("""SELECT periodo, descarga_bit_L, rf_L, diferencia_L, pct_discrepancia,
              dias_faltantes, dias_anomalos, pct_continuidad
       FROM conciliacion_mes ORDER BY pct_discrepancia DESC LIMIT 15""")

,periodo,descarga_bit_L,rf_L,diferencia_L,pct_discrepancia,dias_faltantes,dias_anomalos,pct_continuidad
0,Celaya-2026-03,495000.0,947000,-452000.0,47.73,0,0,100.00
1,Celaya-2026-04,649000.0,1119000,-470000.0,42.00,0,1,100.00
2,Celaya-2026-08,745000.0,1024000,-279000.0,27.25,0,0,100.00
3,Tapachula-2026-05,94000.0,76000,18000.0,23.68,0,5,100.00
4,Cuautitlán-2026-07,16325000.0,13474000,2851000.0,21.16,0,0,100.00
5,Celaya-2026-07,761000.0,963000,-202000.0,20.98,0,0,100.00
6,Celaya-2026-06,795000.0,996000,-201000.0,20.18,0,1,100.00
7,Veracruz-2026-05,385000.0,326000,59000.0,18.10,0,1,100.00
8,Cuernavaca-2026-07,3427000.0,4172000,-745000.0,17.86,4,0,87.10
9,Celaya-2026-05,1006000.0,1223000,-217000.0,17.74,0,1,100.00
